In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import urllib3
from langdetect import detect, DetectorFactory

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
DetectorFactory.seed = 0

#FILTER BAHASA INDONESIA
#Mengevaluasi teks menggunakan pustaka langdetect dan perhitungan stop words.
#penggunaan istilah teknis bahasa Inggris yang lazim terdapat pada abstrak ilmiah.

def is_strictly_indonesian(text):
    try:
        if detect(text) != 'id':
            return False
    except:
        return False
        
    text_lower = " " + text.lower().replace('\n', ' ') + " "
    
    eng_stop_words = [' the ', ' and ', ' of ', ' in ', ' to ', ' is ', ' are ', ' for ', ' with ']
    ind_stop_words = [' yang ', ' dan ', ' di ', ' dengan ', ' untuk ', ' dari ', ' pada ', ' dalam ']
    
    eng_score = sum(text_lower.count(w) for w in eng_stop_words)
    ind_score = sum(text_lower.count(w) for w in ind_stop_words)
    
    if eng_score > 6 or ind_score < 2:
        return False
        
    return True

#PENGUMPULAN DATA MENTAH (OAI-PMH SCRAPING)
# Mengambil metadata abstrak dari daftar portal jurnal terakreditasi SINTA 3.
# Filter kriteria: Tahun terbit <= 2017 dan batas minimal panjang teks > 200 karakter.

def scrape_oai_unlimited_raw(oai_targets):
    all_results = []
    print("-> Memulai proses ekstraksi data mentah...")
    
    for journal_name, oai_url in oai_targets.items():
        print(f"\n-> Memproses jurnal: {journal_name}")
        
        params = {'verb': 'ListRecords', 'metadataPrefix': 'oai_dc'}
        count = 0
        page = 1
        
        while True: 
            try:
                time.sleep(2) 
                response = requests.get(oai_url, params=params, timeout=30, verify=False)
                soup = BeautifulSoup(response.content, 'xml')
                
                records = soup.find_all('record')
                if not records:
                    print(f"Tidak ditemukan record abstrak tambahan.")
                    break
                    
                for record in records:
                    dates = record.find_all('dc:date')
                    tahun_valid = False
                    tahun = None
                    for d in dates:
                        match = re.search(r'\d{4}', d.text)
                        if match:
                            t = int(match.group())
                            if t <= 2017: 
                                tahun_valid = True
                                tahun = t
                                break
                                
                    if not tahun_valid:
                        continue 
                        
                    descriptions = record.find_all('dc:description')
                    for desc in descriptions:
                        teks = desc.text.strip() 
                        
                        # Penyaringan berbasis kepadatan informasi linguistik
                        if len(teks) > 150: 
                            if is_strictly_indonesian(teks):
                                all_results.append({
                                    'source': journal_name,
                                    'year': tahun,
                                    'abstract': teks,
                                    'label': 'human'
                                })
                                count += 1
                                
                                if count % 10 == 0:
                                    print(f"Berhasil mengekstrak {count} abstrak valid...")
                                
                                break 

                token_tag = soup.find('resumptionToken')
                if token_tag and token_tag.text:
                    print(f"Mengakses halaman {page + 1}...")
                    params = {'verb': 'ListRecords', 'resumptionToken': token_tag.text}
                    page += 1
                else:
                    print(f"-> Ekstraksi selesai untuk {journal_name}. Total: {count} abstrak.")
                    break 

            except Exception as e:
                print(f"Terjadi kegagalan koneksi/parsing: {e}")
                break 

    return all_results

daftar_sinta_3 = {
    'JSiI UNSERA': 'https://e-jurnal.lppmunsera.org/index.php/jsii/oai',
    'Sistemasi (Sistem Informasi) UNISI': 'https://sistemasi.ftik.unisi.ac.id/index.php/stmsi/oai',
    'CESS (Computer Engineering, System and Science)': 'https://jurnal.unimed.ac.id/2012/index.php/cess/oai',
    'JISKA (Jurnal Informatika Sunan Kalijaga)': 'https://ejournal.uin-suka.ac.id/saintek/JISKA/oai',
    'ILKOM Jurnal Ilmiah UMI Makassar': 'https://jurnal.fikom.umi.ac.id/index.php/ILKOM/oai',
    'Jurnal Komputer Terapan PCR': 'https://jurnal.pcr.ac.id/index.php/jkt/oai',
    'Jurnal Sisfokom ISB Atma Luhur': 'https://jurnal.atmaluhur.ac.id/index.php/sisfokom/oai',
    'Jurnal COREIT UIN SUSKA': 'http://ejournal.uin-suska.ac.id/index.php/coreit/oai',
    'ELTIKOM (Elektro, Telekomunikasi dan Komputer) POLIBAN': 'https://eltikom.poliban.ac.id/index.php/eltikom/oai',
    'EXPLORE (Jurnal Sistem Informasi dan Telematika) UBL': 'http://jurnal.ubl.ac.id/index.php/explore/oai',
    'JTIP (Jurnal Teknologi Informasi dan Pendidikan) UNP': 'http://tip.ppj.unp.ac.id/index.php/tip/oai',
    'Telematika Jurnal Informatika UPNYK': 'https://jurnal.upnyk.ac.id/index.php/telematika/oai',
    'AITI Jurnal Teknologi Informasi UKSW': 'https://ejournal.uksw.edu/aiti/oai',
    'Jurnal Tekno Kompak TEKNOKRAT': 'https://ejurnal.teknokrat.ac.id/index.php/teknokompak/oai',
    'SIMETRIS (Jurnal Teknik Mesin, Elektro dan Ilmu Komputer) UMK': 'https://jurnal.umk.ac.id/index.php/simet/oai',
    'JIP (Jurnal Informatika Polinema)': 'https://jurnal.polinema.ac.id/index.php/jip/oai',
    'Jurnal TEKNOINFO Teknokrat': 'https://ejurnal.teknokrat.ac.id/index.php/teknoinfo/oai',
    'JUSIFO UIN Raden Fatah': 'https://jurnal.radenfatah.ac.id/index.php/jusifo/oai',
    'INTI Nusa Mandiri': 'https://ejournal.nusamandiri.ac.id/index.php/inti/oai',
    'Jurnal INFOTEL IT Telkom': 'https://ejournal.ittelkom-pwt.ac.id/index.php/infotel/oai',
}

data_human = scrape_oai_unlimited_raw(daftar_sinta_3)

#MENGHAPUS DUPLIKAT DAN PENYIMPANAN DATA 
if data_human:
    df = pd.DataFrame(data_human)
    df.drop_duplicates(subset=['abstract'], inplace=True)
    df.to_csv('../data/raw/dataset_human_pre2018_strict_indo_7_fix.csv', index=False, encoding='utf-8')
    print(f"\n-> Proses eksekusi selesai. Total {len(df)} abstrak berhasil disimpan pada dataset_human_pre2018_strict_indo_7_fix.csv.")
else:
    print("\nTidak ada data yang memenuhi kriteria untuk ditarik.")

-> Memulai proses ekstraksi data mentah...

-> Memproses jurnal: JSiI UNSERA
Tidak ditemukan record abstrak tambahan.

-> Memproses jurnal: Sistemasi (Sistem Informasi) UNISI
Mengakses halaman 2...
Berhasil mengekstrak 10 abstrak valid...
Mengakses halaman 3...
Berhasil mengekstrak 20 abstrak valid...
Mengakses halaman 4...
Berhasil mengekstrak 30 abstrak valid...
Mengakses halaman 5...
Berhasil mengekstrak 40 abstrak valid...
Mengakses halaman 6...
Mengakses halaman 7...
Berhasil mengekstrak 50 abstrak valid...
Mengakses halaman 8...
Berhasil mengekstrak 60 abstrak valid...
Mengakses halaman 9...
Berhasil mengekstrak 70 abstrak valid...
Mengakses halaman 10...
Berhasil mengekstrak 80 abstrak valid...
Mengakses halaman 11...
-> Ekstraksi selesai untuk Sistemasi (Sistem Informasi) UNISI. Total: 88 abstrak.

-> Memproses jurnal: CESS (Computer Engineering, System and Science)
Berhasil mengekstrak 10 abstrak valid...
Berhasil mengekstrak 20 abstrak valid...
Berhasil mengekstrak 30 abstrak

In [3]:
import pandas as pd

df = pd.read_csv('../data/raw/dataset_human_pre2018_strict_indo_7_fix.csv')

#Hitung distribusi berdasarkan kolom 'source' (Nama Jurnal)
distribusi = df['source'].value_counts().reset_index()

#Ubah nama kolom biar enak dibaca
distribusi.columns = ['Nama Jurnal', 'Jumlah Abstrak']

#Tambahkan nomor urut mulai dari 1
distribusi.index = distribusi.index + 1

#Tampilkan hasil
print("Distribusi Hasil Scraping (Data Mentah):")
print("-" * 70)
print(distribusi.to_string())
print("-" * 70)

#Tampilkan total keseluruhan
print(f"Total Portal Jurnal yang Berhasil Terambil : {len(distribusi)} Jurnal")
print(f"Total Keseluruhan Abstrak                  : {distribusi['Jumlah Abstrak'].sum()} Abstrak")

Distribusi Hasil Scraping (Data Mentah):
----------------------------------------------------------------------
                                                      Nama Jurnal  Jumlah Abstrak
1   SIMETRIS (Jurnal Teknik Mesin, Elektro dan Ilmu Komputer) UMK             312
2                                        Jurnal INFOTEL IT Telkom             171
3                               JIP (Jurnal Informatika Polinema)             131
4                              Sistemasi (Sistem Informasi) UNISI              82
5                                  Jurnal Sisfokom ISB Atma Luhur              78
6            EXPLORE (Jurnal Sistem Informasi dan Telematika) UBL              62
7                                ILKOM Jurnal Ilmiah UMI Makassar              61
8                                     Jurnal Komputer Terapan PCR              52
9                                          JUSIFO UIN Raden Fatah              35
10                CESS (Computer Engineering, System and Science)   